In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, joblib
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

In [ ]:
# Load dataset (falls back to a small built-in sample if offline)
try:
    cats = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.mideast"]
    data = fetch_20newsgroups(subset="all", categories=cats, remove=("headers","footers","quotes"))
    X, y, names = data.data, data.target, data.target_names
except Exception:
    names = ["sports", "tech", "food"]
    base = {
        0: ["the team won the match", "great goal in the final game", "the player scored twice", "coach praised the squad", "league title race heats up", "he ran a fast marathon"],
        1: ["new software update released", "the processor is very fast", "machine learning model trained", "the app crashes on startup", "cloud servers scale automatically", "the laptop has a great display"],
        2: ["the pasta was delicious", "bake the cake for an hour", "fresh spices improve the curry", "the restaurant serves great pizza", "add salt to the soup", "grilled fish with lemon"],
    }
    X = [t for k, v in base.items() for t in v] * 10
    y = np.array([k for k, v in base.items() for _ in v] * 10)
print(len(X), "documents |", names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_features=50000)),
    ("svm", LinearSVC(C=1.0)),
])

t0 = time.time()
pipe.fit(X_train, y_train)
print(f"Training time: {time.time()-t0:.2f}s")

pred = pipe.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=names))

In [ ]:
# Quick hyperparameter tuning (fast: linear kernel, 3-fold CV, parallel)
grid = GridSearchCV(pipe, {"svm__C": [0.1, 0.5, 1, 5], "tfidf__ngram_range": [(1, 1), (1, 2)]},
                    cv=3, n_jobs=-1, scoring="accuracy")
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("CV accuracy:", round(grid.best_score_, 4))
best = grid.best_estimator_
print("Test accuracy:", round(accuracy_score(y_test, best.predict(X_test)), 4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, best.predict(X_test), display_labels=names, xticks_rotation=45, cmap="Blues")
plt.title("SVM Text Classification - Confusion Matrix")
plt.tight_layout(); plt.show()

In [ ]:
# Top features per class
tfidf, svm = best.named_steps["tfidf"], best.named_steps["svm"]
feats = np.array(tfidf.get_feature_names_out())
coefs = svm.coef_ if svm.coef_.shape[0] > 1 else np.vstack([-svm.coef_, svm.coef_])
for i, n in enumerate(names):
    print(f"{n}: {', '.join(feats[np.argsort(coefs[i])[-8:][::-1]])}")

In [ ]:
# Predict on new text and save the model
samples = ["NASA launched a new satellite into orbit", "The goalie made an amazing save in overtime",
           "This GPU renders 3D images quickly", "Peace talks in the region continue"]
if len(names) == 3:
    samples = ["The striker scored in the last minute", "The new GPU runs neural networks fast", "Simmer the sauce with garlic"]
for s, p in zip(samples, best.predict(samples)):
    print(f"{names[p]:>25} <- {s}")

joblib.dump(best, "svm_text_classifier.joblib")
print("Model saved: svm_text_classifier.joblib")